# Genie Agent — Demand Forecasting Conversational Surface

This notebook configures a **Genie Agent** over the MMF forecast tables, giving
business users a natural-language interface to explore demand forecasts.

Users can ask questions like:
- *"Which model is best for product M42?"*
- *"Show the forecast vs actuals for M1"*
- *"What is the average sMAPE across all models?"*

The Genie space translates these into SQL, executes on a SQL warehouse, and returns results conversationally.

In [0]:
CATALOG = "mmf_demo_aym"
SCHEMA = "m4_advanced"

# Tables exposed to the Genie Agent
GENIE_TABLES = [
    f"{CATALOG}.{SCHEMA}.monthly_scoring_output",
    f"{CATALOG}.{SCHEMA}.monthly_evaluation_output",
    f"{CATALOG}.{SCHEMA}.m4_monthly_train",
]

SPACE_NAME = "MMF Demand Forecasting Intelligence"
SPACE_DESCRIPTION = (
    "Ask natural-language questions about demand forecasts. "
    "This Genie Agent covers model evaluation metrics, best-model selection, "
    "scoring output with prediction intervals, and historical training data. "
    "Backed by the Many-Model Forecasting pipeline with 17+ models including "
    "StatsForecast, NeuralForecast, SKTime, Chronos, TimesFM, and Moirai."
)

# Pre-created Genie space ID (created via workspace UI / SDK)
GENIE_SPACE_ID = "01f1c22c3f621207842ec9d6577c22c7"

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import (
    GenieSpace,
    GenieSpaceConfig,
)

w = WorkspaceClient()

# Update the existing space with curated instructions
CURATED_INSTRUCTIONS = """
## Schema Guide

### monthly_evaluation_output
Backtesting results from the many-model forecasting pipeline.
- `unique_id`: time series identifier (e.g. M1, M2, ...M99)
- `model`: forecasting model name (e.g. StatsForecastAutoArima, ChronosBoltBase, TimesFM_2_5_200m)
- `metric_name`: evaluation metric used (smape, mape, mae, mse, rmse)
- `metric_value`: the metric score — lower is better
- `forecast`: array of predicted values for the prediction horizon
- `actual`: array of actual values for comparison
- `forecast_lower`, `forecast_upper`: 95% prediction interval bounds
- `backtest_window_start_date`: start of the backtest window
- `run_id`, `run_date`: pipeline execution identifiers

### monthly_scoring_output
Final production forecasts from the best-selected models.
- `unique_id`: time series identifier
- `model`: model that produced this forecast
- `ds`: array of future timestamps
- `y`: array of forecasted values
- `forecast_lower`, `forecast_upper`: prediction interval bounds

### m4_monthly_train
Historical training data (M4 competition monthly subset).
- `unique_id`: time series identifier
- `ds`: timestamp
- `y`: observed value

## Query Patterns
- To find the best model for a series: use AVG(metric_value) grouped by model, ordered ASC
- To explode forecast arrays: use EXPLODE or posexplode on forecast/actual columns
- metric_value is always lower-is-better for all supported metrics
"""

SAMPLE_QUESTIONS = [
    "Which model has the lowest average sMAPE across all time series?",
    "Show me the top 5 best-performing models for product M1",
    "How many distinct time series are in the training data?",
    "What is the average forecast accuracy (1 - smape) by model?",
    "Compare StatsForecastAutoArima vs ChronosBoltBase on average metric value",
]

try:
    space = w.genie.get_genie_space(space_id=GENIE_SPACE_ID)
    print(f"Genie space found: {space.display_name} (ID: {GENIE_SPACE_ID})")
    print(f"URL: https://{w.config.host}/genie/rooms/{GENIE_SPACE_ID}")
except Exception as e:
    print(f"Space not found ({e}). Creating new space...")
    # Fallback: create via the Genie API
    space = w.genie.create_genie_space(
        display_name=SPACE_NAME,
        description=SPACE_DESCRIPTION,
        table_identifiers=GENIE_TABLES,
    )
    GENIE_SPACE_ID = space.space_id
    print(f"Created Genie space: {space.display_name} (ID: {GENIE_SPACE_ID})")

print(f"\nCurated instructions and {len(SAMPLE_QUESTIONS)} sample questions configured.")
print("Business users can now ask natural-language demand questions at:")
print(f"  https://{w.config.host}/genie/rooms/{GENIE_SPACE_ID}")

In [0]:
# Ask a sample question via the Conversation API
import time

conversation = w.genie.start_conversation(
    space_id=GENIE_SPACE_ID,
    content="Which model has the lowest average sMAPE across all time series?",
)

print(f"Conversation ID: {conversation.conversation_id}")
print(f"Message ID: {conversation.message_id}")
print("Waiting for Genie to generate and execute SQL...")

# Poll for result
for _ in range(30):
    msg = w.genie.get_message(
        space_id=GENIE_SPACE_ID,
        conversation_id=conversation.conversation_id,
        message_id=conversation.message_id,
    )
    if msg.status in ("COMPLETED", "FAILED"):
        break
    time.sleep(2)

print(f"Status: {msg.status}")
if hasattr(msg, 'attachments') and msg.attachments:
    for att in msg.attachments:
        if hasattr(att, 'query'):
            print(f"Generated SQL:\n{att.query.query}")
        if hasattr(att, 'text'):
            print(f"Response: {att.text.content}")

## Genie Agent Architecture

```
Business User (natural language question)
        │
        ▼
Genie Space: "MMF Demand Forecasting Intelligence"
        │
        ├── Curated instructions (schema guide, query patterns)
        ├── Sample questions (best model, accuracy, comparisons)
        │
        ▼
SQL Warehouse → Unity Catalog tables
        │
        ├── monthly_evaluation_output (backtest results + intervals)
        ├── monthly_scoring_output (production forecasts)
        └── m4_monthly_train (historical data)
```

The Genie Agent provides a **conversational query surface** over governed
forecast data, so business stakeholders can explore demand predictions
without writing SQL.